# Pneumonia Detection - Custom CNN with Performance Optimizations
This notebook trains a custom CNN to detect pneumonia from chest X-ray images with improved architecture and techniques to exceed 90% accuracy.

In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from datetime import datetime
import pickle

# =============================================================================
# CONFIGURATION - OPTIMIZED FOR BEST ACCURACY
# =============================================================================
IMG_SIZE = 224
BATCH_SIZE = 16          # Smaller batch size for better gradient updates
EPOCHS = 100            # More epochs - will stop early if no improvement
INITIAL_LR = 0.001      # Higher learning rate for faster convergence
PATIENCE_EARLY = 20     # More patience to find optimal point
PATIENCE_LR = 10        # More patience for LR reduction


BASE_DIR = 'chest_xray'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR   = os.path.join(BASE_DIR, 'val')
TEST_DIR  = os.path.join(BASE_DIR, 'test')

# Output directories
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

# =============================================================================
# DATA GENERATORS - AGGRESSIVE AUGMENTATION FOR BETTER GENERALIZATION
# =============================================================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,              # More rotation
    width_shift_range=0.2,          # More horizontal shift
    height_shift_range=0.2,         # More vertical shift
    shear_range=0.15,               # More shearing
    zoom_range=0.25,                # More zoom variation
    horizontal_flip=True,
    vertical_flip=True,             # NEW: Vertical flips for X-rays
    brightness_range=[0.7, 1.3],    # NEW: Brightness variations
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['NORMAL', 'PNEUMONIA'],
    shuffle=True,
    seed=42
)

validation_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['NORMAL', 'PNEUMONIA'],
    shuffle=False,
    seed=42
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['NORMAL', 'PNEUMONIA'],
    shuffle=False,
    seed=42
)

print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Test samples: {test_generator.samples}")

# =============================================================================
# CALCULATE CLASS WEIGHTS - Handle imbalanced data
# =============================================================================
class_weights = compute_class_weight(
    'balanced',
    classes=np.array([0, 1]),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))
print(f"\nClass weights for imbalance handling:")
print(f"  NORMAL: {class_weight_dict[0]:.3f}")
print(f"  PNEUMONIA: {class_weight_dict[1]:.3f}")

# =============================================================================
# BUILD OPTIMIZED CNN ARCHITECTURE (Best Custom Model)
# =============================================================================
print("\n" + "="*70)
print("BUILDING OPTIMIZED CUSTOM CNN")
print("="*70)

model = Sequential([
    # Block 1 - Enhanced feature extraction
    Conv2D(64, (3,3), padding='same', kernel_regularizer=l2(0.001), input_shape=(IMG_SIZE,IMG_SIZE,3)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(64, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Block 2 - Deeper feature learning
    Conv2D(128, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(128, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Block 3 - Medium features
    Conv2D(256, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(256, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Block 4 - High-level features
    Conv2D(512, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(512, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Global Average Pooling
    GlobalAveragePooling2D(),

    # Dense layers with BatchNorm
    Dense(512, kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),
    
    Dense(256, kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.4),
    
    Dense(128, kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.3),
    
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=INITIAL_LR),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()
print("\n✅ Optimized Custom CNN Created!")

# =============================================================================
# CALLBACKS - Better training control
# =============================================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_path = f'models/pneumonia_cnn_best_{timestamp}.h5'

callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=PATIENCE_EARLY,
        restore_best_weights=True,
        mode='max',
        verbose=1,
        min_delta=0.001
    ),

    ModelCheckpoint(
        checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=PATIENCE_LR,
        mode='max',
        verbose=1,
        min_lr=1e-7
    )
]

# =============================================================================
# TRAINING WITH CLASS WEIGHTS
# =============================================================================
print("\n" + "="*70)
print("TRAINING OPTIMIZED CUSTOM CNN")
print("="*70)

history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight_dict,  # Handle class imbalance
    verbose=1
)

print("\n✅ Training completed!")

# =============================================================================
# EVALUATION ON TEST SET
# =============================================================================
best_model = tf.keras.models.load_model(checkpoint_path)

test_generator.reset()
predictions = best_model.predict(test_generator, verbose=1)
predicted_classes = (predictions > 0.5).astype(int).flatten()
true_classes = test_generator.classes

test_acc = accuracy_score(true_classes, predicted_classes)
cm = confusion_matrix(true_classes, predicted_classes)
tn, fp, fn, tp = cm.ravel()

print(f"\n{'='*70}")
print(f"TEST ACCURACY: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"{'='*70}")
print("\nConfusion Matrix:")
print(cm)

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
print(f"\nSensitivity (Pneumonia Detection Rate): {sensitivity:.4f} ({sensitivity*100:.2f}%)")
print(f"Specificity (Normal Detection Rate): {specificity:.4f} ({specificity*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(true_classes, predicted_classes,
                            target_names=['NORMAL', 'PNEUMONIA']))

# =============================================================================
# PLOT RESULTS
# =============================================================================
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
plt.axhline(y=test_acc, color='r', linestyle='--', label=f'Test Acc: {test_acc:.2%}')
plt.title('Accuracy over Epochs', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['NORMAL','PNEUMONIA'],
            yticklabels=['NORMAL','PNEUMONIA'])
plt.title(f'Confusion Matrix (Accuracy: {test_acc:.2%})', fontsize=12, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('True')

plt.tight_layout()
plt.savefig(f'results/accuracy_report_{timestamp}.png', dpi=100)
plt.show()

# =============================================================================
# SAVE FINAL REPORT
# =============================================================================
report = f"""
{'='*75}
                    OPTIMIZED CUSTOM CNN - FINAL REPORT
{'='*75}
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Model: Custom CNN with Batch Normalization & L2 Regularization

TEST ACCURACY: {test_acc:.4f} ({test_acc*100:.2f}%)

PERFORMANCE METRICS:
  - Sensitivity: {sensitivity:.4f} ({sensitivity*100:.2f}%)
  - Specificity: {specificity:.4f} ({specificity*100:.2f}%)
  - True Positives: {tp}
  - True Negatives: {tn}
  - False Positives: {fp}
  - False Negatives: {fn}

MODEL DETAILS:
  - Total Parameters: {best_model.count_params():,}
  - Training Epochs: {len(history.history['accuracy'])}
  - Model File: {checkpoint_path}
  - Batch Size: {BATCH_SIZE}
  - Learning Rate: {INITIAL_LR}

IMPROVEMENTS IMPLEMENTED:
  1. Batch Normalization for stable training
  2. L2 Regularization (0.001) to prevent overfitting
  3. Aggressive data augmentation (40° rotation, 30% zoom, brightness)
  4. Class weight balancing for imbalanced data
  5. Optimized hyperparameters (16 batch size, 0.001 LR)
  6. Dropout regularization at strategic points
  7. More conv blocks for feature extraction
  8. Global Average Pooling for dimensionality reduction

{'='*75}
"""

print(report)
with open(f'results/final_report_{timestamp}.txt', 'w') as f:
    f.write(report)

# Save as main model for production
import shutil
shutil.copy(checkpoint_path, 'models/pneumonia_cnn_best.h5')
print(f"\n✅ Model saved as: models/pneumonia_cnn_best.h5")
print(f"✅ All results saved in 'results/' folder")


c:\Users\janet taguta\Desktop\PROJECT\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


c:\Users\janet taguta\Desktop\PROJECT\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Training samples: 5216
Validation samples: 16
Test samples: 624

Class weights for imbalance handling:
  NORMAL: 1.945
  PNEUMONIA: 0.673

BUILDING OPTIMIZED CUSTOM CNN


c:\Users\janet taguta\Desktop\PROJECT\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Training samples: 5216
Validation samples: 16
Test samples: 624

Class weights for imbalance handling:
  NORMAL: 1.945
  PNEUMONIA: 0.673

BUILDING OPTIMIZED CUSTOM CNN


...

 Trainable params: 5,118,017 (19.52 MB)

 Non-trainable params: 5,632 (22.00 KB)


✅ Optimized Custom CNN Created!

LOADING BEST MODEL (83% ACCURACY AT 50 EPOCHS)
Loading model from: models/pneumonia_cnn_best.h5



✅ Model loaded successfully!
Model has 2,437,825 parameters


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'models/pneumonia_cnn_best_20260302_173446.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
# =============================================================================
# IMPROVED CNN MODEL WITH BATCH NORMALIZATION & BETTER ARCHITECTURE
# =============================================================================
# This model uses:
# 1. Batch Normalization (stabilizes training)
# 2. More filters in early layers
# 3. L2 regularization
# 4. Better activation patterns
# 5. Optimized dropout placement

from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.regularizers import l2

model = Sequential([
    # Block 1
    Conv2D(64, (3,3), padding='same', kernel_regularizer=l2(0.001), input_shape=(IMG_SIZE,IMG_SIZE,3)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(64, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Block 2
    Conv2D(128, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(128, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Block 3
    Conv2D(256, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(256, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Block 4
    Conv2D(512, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(512, (3,3), padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Global Average Pooling
    GlobalAveragePooling2D(),

    # Dense layers with BatchNorm
    Dense(512, kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),
    
    Dense(256, kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.4),
    
    Dense(128, kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.3),
    
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()
print("\n✅ Improved Custom CNN Model Created!")

In [ ]:
# =============================================================================
# IMPROVED DATA AUGMENTATION (More aggressive)
# =============================================================================
train_datagen_improved = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,              # Increased from 20
    width_shift_range=0.2,          # Increased from 0.1
    height_shift_range=0.2,         # Increased from 0.1
    shear_range=0.15,               # Increased from 0.05
    zoom_range=0.2,                 # Increased from 0.1
    horizontal_flip=True,
    vertical_flip=True,             # NEW: Add vertical flips for X-rays
    brightness_range=[0.8, 1.2],    # NEW: Brightness variations
    fill_mode='nearest'
)

val_test_datagen_improved = ImageDataGenerator(rescale=1./255)

# Create generators with improved augmentation
train_generator_improved = train_datagen_improved.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=16,                  # Smaller batch size for better gradients
    class_mode='binary',
    classes=['NORMAL', 'PNEUMONIA'],
    shuffle=True,
    seed=42
)

validation_generator_improved = val_test_datagen_improved.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=16,
    class_mode='binary',
    classes=['NORMAL', 'PNEUMONIA'],
    shuffle=False,
    seed=42
)

test_generator_improved = val_test_datagen_improved.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=16,
    class_mode='binary',
    classes=['NORMAL', 'PNEUMONIA'],
    shuffle=False,
    seed=42
)

print("✅ Improved data generators created with enhanced augmentation!")
print(f"Training samples: {train_generator_improved.samples}")
print(f"Validation samples: {validation_generator_improved.samples}")
print(f"Test samples: {test_generator_improved.samples}")

# =============================================================================
# CONFIGURATION FOR IMPROVED TRAINING
# =============================================================================
BATCH_SIZE_IMPROVED = 16
EPOCHS_IMPROVED = 80
INITIAL_LR_IMPROVED = 0.001
PATIENCE_EARLY_IMPROVED = 20
PATIENCE_LR_IMPROVED = 8

In [ ]:
# =============================================================================
# IMPROVED CALLBACKS FOR BETTER TRAINING
# =============================================================================
from tensorflow.keras.callbacks import ReduceLROnPlateau, LearningRateScheduler

timestamp_improved = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_path_improved = f'models/pneumonia_cnn_improved_{timestamp_improved}.h5'

# Learning rate scheduler for gradual reduction
def lr_schedule(epoch, lr):
    if epoch < 20:
        return lr
    elif epoch < 40:
        return lr * 0.8
    elif epoch < 60:
        return lr * 0.5
    else:
        return lr * 0.2

callbacks_improved = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=PATIENCE_EARLY_IMPROVED,
        restore_best_weights=True,
        mode='max',
        verbose=1,
        min_delta=0.001
    ),
    
    ModelCheckpoint(
        checkpoint_path_improved,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    
    ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=PATIENCE_LR_IMPROVED,
        mode='max',
        verbose=1,
        min_lr=1e-7
    ),
    
    LearningRateScheduler(lr_schedule, verbose=1)
]

print("✅ Improved callbacks configured!")

# =============================================================================
# TRAIN IMPROVED MODEL
# =============================================================================
print("\n" + "="*70)
print("TRAINING IMPROVED CUSTOM CNN MODEL")
print("="*70)

history_improved = model.fit(
    train_generator_improved,
    steps_per_epoch=train_generator_improved.samples // BATCH_SIZE_IMPROVED,
    epochs=EPOCHS_IMPROVED,
    validation_data=validation_generator_improved,
    validation_steps=validation_generator_improved.samples // BATCH_SIZE_IMPROVED,
    callbacks=callbacks_improved,
    verbose=1
)

print("\n✅ Training completed!")

In [ ]:
# =============================================================================
# EVALUATE IMPROVED MODEL ON TEST SET
# =============================================================================
# Load best improved model
best_model_improved = tf.keras.models.load_model(checkpoint_path_improved)

# Reset test generator
test_generator_improved.reset()
predictions_improved = best_model_improved.predict(test_generator_improved, verbose=1)
predicted_classes_improved = (predictions_improved > 0.5).astype(int).flatten()
true_classes_improved = test_generator_improved.classes

# Calculate metrics
test_acc_improved = accuracy_score(true_classes_improved, predicted_classes_improved)
print(f"\n{'='*70}")
print(f"TEST ACCURACY (IMPROVED MODEL): {test_acc_improved:.4f} ({test_acc_improved*100:.2f}%)")
print(f"{'='*70}")

# Confusion Matrix
cm_improved = confusion_matrix(true_classes_improved, predicted_classes_improved)
tn_improved, fp_improved, fn_improved, tp_improved = cm_improved.ravel()

print("\nConfusion Matrix:")
print(cm_improved)
print(f"\nTrue Positives (Detected Pneumonia): {tp_improved}")
print(f"True Negatives (Detected Normal): {tn_improved}")
print(f"False Positives (Normal as Pneumonia): {fp_improved}")
print(f"False Negatives (Pneumonia as Normal): {fn_improved}")

# Classification Report
print("\nDetailed Classification Report:")
print(classification_report(true_classes_improved, predicted_classes_improved,
                            target_names=['NORMAL', 'PNEUMONIA']))

# Calculate sensitivity and specificity
sensitivity = tp_improved / (tp_improved + fn_improved)
specificity = tn_improved / (tn_improved + fp_improved)
print(f"\nSensitivity (True Positive Rate): {sensitivity:.4f}")
print(f"Specificity (True Negative Rate): {specificity:.4f}")

# =============================================================================
# PLOT TRAINING RESULTS
# =============================================================================
plt.figure(figsize=(14,5))

# Accuracy curves
plt.subplot(1,2,1)
plt.plot(history_improved.history['accuracy'], label='Train Accuracy', linewidth=2)
plt.plot(history_improved.history['val_accuracy'], label='Val Accuracy', linewidth=2)
plt.axhline(y=test_acc_improved, color='r', linestyle='--', label=f'Test Acc: {test_acc_improved:.2%}')
plt.title('Accuracy over Epochs', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# Loss curves
plt.subplot(1,2,2)
plt.plot(history_improved.history['loss'], label='Train Loss', linewidth=2)
plt.plot(history_improved.history['val_loss'], label='Val Loss', linewidth=2)
plt.title('Loss over Epochs', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'results/accuracy_curves_improved_{timestamp_improved}.png', dpi=100)
plt.show()

# =============================================================================
# PLOT CONFUSION MATRIX
# =============================================================================
plt.figure(figsize=(8,6))
sns.heatmap(cm_improved, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['NORMAL','PNEUMONIA'],
            yticklabels=['NORMAL','PNEUMONIA'],
            annot_kws={'size': 14, 'fontweight': 'bold'})
plt.title('Confusion Matrix - Improved Model (Test Set)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted', fontweight='bold')
plt.ylabel('True', fontweight='bold')
plt.tight_layout()
plt.savefig(f'results/confusion_matrix_improved_{timestamp_improved}.png', dpi=100)
plt.show()

# =============================================================================
# PERFORMANCE REPORT
# =============================================================================
benchmarks = {
    'Kermany et al. (2018)': 0.9280,
    'Stephen et al. (2019)': 0.9300,
    'Becker (2022)': 0.9000
}

report_improved = f"""
{'='*75}
                  PNEUMONIA DETECTION - IMPROVED CNN
                     FINAL PERFORMANCE REPORT
{'='*75}
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Model: Custom CNN with Batch Normalization, L2 Regularization & Data Augmentation

TEST ACCURACY: {test_acc_improved:.4f} ({test_acc_improved*100:.2f}%)

PERFORMANCE METRICS:
  - Sensitivity (True Positive Rate): {sensitivity:.4f} ({sensitivity*100:.2f}%)
  - Specificity (True Negative Rate): {specificity:.4f} ({specificity*100:.2f}%)
  - True Positives (Pneumonia Detected Correctly): {tp_improved}
  - True Negatives (Normal Detected Correctly): {tn_improved}
  - False Positives (Normal misclassified as Pneumonia): {fp_improved}
  - False Negatives (Pneumonia misclassified as Normal): {fn_improved}

COMPARISON WITH LITERATURE BENCHMARKS:
"""

for name, acc in benchmarks.items():
    status = "✓ EXCEEDS" if test_acc_improved > acc else "✗ BELOW" if test_acc_improved < acc else "= MATCHES"
    report_improved += f"  {name}: {acc:.4f} → {status}\n"

report_improved += f"""
MODEL ARCHITECTURE:
  - Total Parameters: {best_model_improved.count_params():,}
  - Core Features: Batch Normalization, L2 Regularization, Multiple Conv Blocks
  - Training Epochs: {len(history_improved.history['accuracy'])}
  - Model File: {checkpoint_path_improved}

IMPROVEMENTS MADE:
  1. Batch Normalization for stable training
  2. L2 Regularization to prevent overfitting
  3. Enhanced data augmentation (rotation, zoom, brightness, flips)
  4. Learning rate scheduling
  5. Smaller batch size (16) for better gradient updates
  6. More convolutional blocks with regularization
  7. Additional dense layers with batch norm

{'='*75}
"""

print(report_improved)

# Save report
with open(f'results/performance_report_improved_{timestamp_improved}.txt', 'w') as f:
    f.write(report_improved)

print(f"\n✅ All results saved in 'results/' folder!")
print(f"✅ Model saved: {checkpoint_path_improved}")